In [1]:
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool, ToolRuntime
from langchain.agents import AgentState
from langgraph.types import Command
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from typing import Callable

In [2]:
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

True

## state
记录用户的邮箱授权状态

In [3]:
class AuthenticatedState(AgentState):
    authenticated: bool

## tool
- authenticate : 用来模拟用户邮箱鉴权
- check_inbox : 模拟收取邮件
- send_email : 模拟发送邮件

In [4]:
# 用户邮箱授权
@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""

    # 定义变量，记录校验结果
    authenticated = False
    message = "Authentication failed"

    # 校验邮箱和密码
    if email == "huge@itcast.cn" and password == "123":
        authenticated = True
        message = "Successfully authenticated"

    # 返回校验结果
    return Command(
        update={
            "authenticated": authenticated,
            "messages": [
                ToolMessage(
                    message,
                    tool_call_id = runtime.tool_call_id
                )
            ]
        }
    )

In [5]:
# 邮箱操作工具
@tool
def check_inbox() -> str:
    """Read an email from the given address."""
    # 模拟收件箱邮件
    return [
        {
            "subject": "周末见个面？",
            "content": """
                嗨 虎哥，
                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？

                祝好，简
            """,
            "from": "jane@itcast.cn",
            "status": "unread"
        },
        {
            "subject": "周五会议",
            "content": """
                嗨 虎哥，
                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？

                祝好，小李
            """,
            "from": "lixiaolong@itcast.cn",
            "status": "checked"
        }
    ]

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"邮件已发送至 {to} , 主题： {subject} , 内容： {body}"

## 动态工具中间件
- 在用户邮箱授权之前，只能看到authenticate工具
- 在用户邮箱授权之后，只能看到send_email和check_inbox工具

In [7]:
@wrap_model_call
def dynamic_tool_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Allow read inbox and send email tools only if user provides correct email and password"""
    # 读取授权状态
    authenticated = request.state.get("authenticated")

    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    # 重写工具列表
    request = request.override(tools=tools)
    return handler(request)

## 动态提示词中间件
- 鉴权阶段：告知AI的核心工作是鉴权
- 工作阶段：告知AI的核心工作是处理邮件

In [8]:
from langchain.agents.middleware import dynamic_prompt

# 授权前，要求鉴定用户权限的系统提示词
unauthenticated_prompt = """You are a helpful email assistant.
    For system security protocols, you must authenticate user before any other interaction."""

# 授权后，邮件处理的系统提示词
authenticated_prompt = "You are a helpful assistant that can check the inbox and send emails."

# 在中间件中通过判断state的authenticated值来动态切换提示词
@dynamic_prompt
def dynamic_prompt_func(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")
    final_prompt = authenticated_prompt if authenticated else unauthenticated_prompt
    return final_prompt

## Agent

In [9]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

checkpoint = InMemorySaver()

agent = create_agent(
    model="deepseek-v4-flash",
    tools=[authenticate, check_inbox, send_email],
    state_schema=AuthenticatedState,
    checkpointer=checkpoint,
    middleware=[
        dynamic_tool_call,
        dynamic_prompt_func,
        HumanInTheLoopMiddleware(
            interrupt_on = {  # 这种写法告诉agent，在调用哪些工具时要人工干预，哪些不用
                "authenticate": False,
                "check_inbox": False,
                "send_email": True
            }
        )
    ]
)

## HITL的stream模式
由于HITL并不是模型的能力，而是LangChain利用wrap_tool_call在模型调用工具前后的一种拦截行为。因此采用messages模式只能看到模型返回的消息，无法看到interrupt中断信息。
在HITL存在时，stream采用的模式比较特殊，需要同时包含两种模式：
- messages : 以token方式返回AI生成的内容
- updates : 返回Agent调用时的每个步骤的完整信息，比如当出现interrupt时，完整展示interrupt信息  


设定stream的version为v2，这样返回的结果格式会以dict返回，而不是tuple，会更好处理。

In [10]:
config = {"configurable": {"thread_id": "2"}}

response = agent.stream(
    {"messages": HumanMessage("帮我查看一下邮件")},
    config = config,
    stream_mode=["messages", "updates"],
    version="v2"
)

for chunk in response:
    print(f"---------------------{chunk["type"]}-----------------------")
    print(chunk["data"])

---------------------messages-----------------------
(AIMessageChunk(content='', additional_kwargs={'reasoning_content': ''}, response_metadata={'model_provider': 'deepseek'}, id='lc_run--019fca71-9c74-7660-8707-ad7af8eede53', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'ls_integration': 'langchain_chat_model', 'thread_id': '2', 'langgraph_step': 1, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:7e1f6a47-64f3-50f0-8352-50b1b8da03db', 'checkpoint_ns': 'model:7e1f6a47-64f3-50f0-8352-50b1b8da03db', 'ls_provider': 'deepseek', 'ls_model_name': 'deepseek-v4-flash', 'ls_model_type': 'chat', 'ls_temperature': None, 'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14', 'langchain-openai': '1.4.1'}})
---------------------messages-----------------------
(AIMessageChunk(content='', additional_kwargs={'reasoning_content': 'The'}, response_metadata={'model_provider': '

## 测试

### 邮箱授权

In [ ]:
from typing import Any

response = agent.stream(
    {"messages": [HumanMessage("huge@itcast.cn/123")]},
    config = config,
    stream_mode=["messages", "updates"],
    version="v2"
)

def print_chunk(chunk: dict[str, Any]):
    type = chunk["type"]
    data = chunk["data"]

    if type == "messages":
        token, metadata = data
        if isinstance(token, AIMessage) and token.content:
            print(token.content, end="", flush=True)
    elif type == "updates":
        print("")
        print("=====================updates==================")
        print(data)

for chunk in response:
    print_chunk(chunk)

### 触发Interrupt

In [ ]:
# 用户授权
response = agent.stream(
    {"messages": [HumanMessage("帮我回复Jane，告诉她很期待她能来，问问具体时间")]},
    config=config,
    stream_mode=["messages", "updates"],
    version="v2"
)

for chunk in response:
    print_chunk(chunk)

### 用户介入

In [ ]:
response = agent.stream(
    Command(
        resume={
            "decision": [
                {
                    "type": "reject",
                    # 告知ai拒绝的原因
                    "messages": "语气太书面了，这不是工作邮件，Jane是我的好朋友，回复自然一点。"
                }
            ]
        }
    ),
    config=config,
    stream_mode=["messages", "updates"],
    version="v2"
)

for chunk in response:
    print_chunk(chunk)

## 会话历史

### checkpoint获取历史
checkpointer中直接获取拿不到interrupt信息，只能拿到Message信息

### state获取历史
如果要同时获取到历史消息、interrupt等信息，必须通过AgentState

In [ ]:
agent.get_state(config)

### 历史消息处理

In [ ]:
def get_messages(thread_id: str) -> dict:
    """获取会话历史，如果存在中断则返回中断信息"""
    config = {"configurable": {"thread_id": thread_id}}

    state = agent.get_state(config)

    if state is None or not state.values:
        return {"messages": []}

    messages = state.values.get("messages", [])

    # 转换消息格式
    result = []
    for msg in messages:
        if not msg.content:
            continue
        if isinstance(msg, HumanMessage):
            result.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            result.append({"role": "assistant", "content": msg.content})


    response = {"messages": result}

    # 检查是否存在中断
    interrupts = None
    if hasattr(state, "interrupts") and state.interrupts:
        interrupt = state.interrupts
    elif hasattr(state, 'tasks') and state.tasks:
        for task in state.tasks:
            if hasattr(task, 'interrupts') and task.interrupts:
                interrupt = task.interrupts
                break

    if interrupts:
        response["has_interrupt"] = True
        response["interrupt"] = {
            "reason": "需要人工确认",
            "detail": _serialize(interrupt)
        }

    return response

def _serialize(obj):
    """递归转换对象为可 JSON 序列化的格式"""
    if hasattr(obj, 'value'):
        return _serialize(obj.value)
    elif hasattr(obj, "model_dump"):
        return obj.model_dump()
    elif isinstance(obj, (list, tuple)):
        return [_serialize(item) for item in obj]
    elif isinstance(obj, dict):
        return {k: _serialize(v) for k, v in obj.items()}

    return obj

## 同步与异步

### 异步流式输出

In [ ]:
# 同步
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "......."}]},
    stream_mode="messages"
):
    token, metadata = chunk
    print(token.text, end="", flush=True)

# 异步
async for chunk in agent.astream(
    {"messages": [{"role": "user", "content": "..........."}]},
    stream_mode="messages"
):
    token, metadata = chunk
    print(token.text, end="", flush=True)

## 异步Checkpointer
一旦使用了异步API，Agent的Checkpointer也必须使用异步版本。

In [ ]:
import aiosqlite
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

async def test():
    # 创建连接
    conn = await aiosqlite.connect("agent.db")
    # 创建checkpointer
    checkpoint = AsyncSqliteSaver(conn=conn)

    # 关闭连接
    await conn.close()

### 重构ChatRequest

### SSE
让服务器主动向浏览器推送数据的通信协议。  
前端发起一次请求，服务端以stream方式不断推送新的数据给前端，前端完成渲染，从而实现大模型的流式输出效果。

标准的SSE中，服务端返回的数据格式是纯文本格式，可以包含event、data、id、retry字段，每个字段后跟冒号和空格，并以\n结束，不同消息之间则以两个\n结束

In [ ]:
[field]: [value]\n
[field]: [value]\n
\n\n


|字段|说明|
|----|----|
|event|指定事件类型（前端可监听特定事件）|
|data|实际数据内容（可以是多行，多行会拼接）|
|id|消息ID，客户端可用于断线重连|
|retry|重连时间（毫秒），客户端断线后等待多久重新连接|